# Chapter 13: Agentic RAG
## Topic 1: Static RAG vs Agentic RAG

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every RAG pipeline built across Chapters 7-9 has followed one fixed shape: retrieve, then generate. Retrieval always runs (or, after Chapter 9 Topic 1's triggering logic, runs based on a rule-based or confidence-based decision made *before* the model ever reasons about the specific query), a fixed number of chunks come back, and generation proceeds over whatever was retrieved — the model itself never participates in deciding whether to retrieve, what to retrieve, or whether to retrieve again if the first attempt wasn't good enough. This fixed, non-negotiable shape is what "static RAG" means precisely.
- **Agentic RAG** removes that fixed shape and gives the model itself a genuine role in the retrieval decision — this is the direct, natural culmination of two things already built separately in this notebook series: Chapter 9 Topic 3's `search_knowledge_base` tool (retrieval exposed as something a model can *choose* to call) and Chapter 10's tool-engineering discipline (making that choice reliable). Agentic RAG isn't a new retrieval technique — it's a different *control structure* wrapped around the same retrieval and generation machinery already built.
- The core reframe this topic makes explicit: static RAG treats retrieval as a mandatory pipeline stage; agentic RAG treats retrieval as one tool among several the model can invoke zero, one, or multiple times, in whatever sequence its own reasoning determines is appropriate for the specific query at hand — including deciding *not* to retrieve at all, or retrieving, evaluating the result, and retrieving again with a refined query if the first attempt was insufficient.


### 2. Internal Working — Step by Step

**The mechanical difference between static and agentic RAG, concretely:**

1. **Static RAG's control flow is fixed and external to the model:** `retrieve() -> construct_prompt() -> generate()`, always in that order, always exactly once, regardless of the specific query's actual needs. The model only ever sees this pipeline's final output — a context block plus a question — and generates a response; it has no opportunity to influence whether retrieval happened, what was retrieved, or whether a second attempt was needed.
2. **Agentic RAG's control flow is decided by the model itself, using exactly the tool-calling mechanics already built in Chapter 10**: the model receives a query and a `search_knowledge_base` tool description (Chapter 9 Topic 3); it decides, based on the query's specific content, whether calling that tool is warranted at all (Chapter 9 Topic 1's triggering logic, now happening *inside* the model's own reasoning rather than as an external rule); if it decides to call it, the standard tool-call request/dispatch/result cycle (Chapter 10 Topic 2) runs exactly as built; and critically, the model can *evaluate the returned result* and decide whether it's sufficient or whether another retrieval call (perhaps with a refined query) is warranted before generating a final answer.
3. **The "evaluate and possibly retry" capability is the genuinely new mechanical piece agentic RAG adds** beyond what Chapter 9 Topic 3 alone already built — Chapter 9 Topic 3 demonstrated a model choosing *whether* to call `search_knowledge_base` once; agentic RAG additionally lets the model reason about the *quality* of what came back and iterate, using the same underlying agent-loop mechanics (Chapter 10 Topic 2's `messages` list, growing with each tool call and result) that already support multi-step tool use for any other reason.
4. **This is not a different retrieval algorithm — Chapter 7's entire hybrid BM25+dense+RRF+reranking pipeline remains completely unchanged underneath.** What changes is purely the control structure deciding *when* and *how many times* that unchanged pipeline gets invoked for a given query, and *whether the model itself* makes that call rather than an external, fixed rule.


### 3. How This Is Implemented in This Project

- This project already has every mechanical piece agentic RAG requires, built separately across earlier chapters: `search_knowledge_base` as a callable tool (Chapter 9 Topic 3), the tool-calling request/dispatch/result mechanics (Chapter 10 Topics 1-2), and a working multi-step agent loop capable of multiple tool calls per turn (Chapter 10 Topic 6). Agentic RAG, in this project's specific context, is the recognition that these pieces, already built and tested independently, *compose* into exactly this pattern without requiring new retrieval infrastructure.
- The concrete difference this project would need to design deliberately: the system prompt's instructions to the model about *evaluating* a `search_knowledge_base` result and deciding whether to retrieve again — this is new guidance beyond what Chapter 9 Topic 6's RAG-specific prompting already covers (which addressed how to use retrieved context once received, not whether the model should judge that context as insufficient and seek more).
- This directly sets up this chapter's later topics: Topic 3's "check the catalog only if uncertain" decision step is a concrete, specific instance of exactly this pattern applied to `lookup_product_catalog` (Chapter 10 Topic 6) rather than `search_knowledge_base`, and Topic 4's Smart Saver FD test (revisiting Chapter 9 Topic 8's proof-of-concept) specifically validates whether this project's agentic triggering decision, not just its grounded-generation quality, actually works correctly for genuinely novel content.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Agentic RAG's flexibility comes with a real, direct cost and latency trade-off compared to static RAG** — every additional evaluate-and-possibly-retry cycle is a full additional model generation call plus another retrieval pipeline invocation, compounding exactly per Chapter 10 Topic 2's message-list growth mechanics. A query that could have been answered correctly by static RAG's single retrieve-then-generate pass costs strictly more under agentic RAG if the model chooses to retrieve multiple times unnecessarily.
- **The model's own judgment about "is this result sufficient" is itself a probabilistic behavior, not a guaranteed one** — exactly the same instruction-following uncertainty already established for tool-calling generally (Chapter 8 Topic 4's hallucination-detection discussion, Chapter 9 Topic 3's measured triggering-accuracy findings). A model that's overconfident about insufficient retrieved context, or unnecessarily cautious about genuinely sufficient context, produces real, measurable quality or cost problems respectively — this needs the same evidence-based validation discipline (a labeled test set, measured accuracy) as every other model-decided behavior in this notebook series, not an assumption that agentic RAG "just works better."
- **Static RAG's predictability is a genuine advantage agentic RAG gives up** — a fixed retrieve-then-generate pipeline has bounded, predictable latency and cost per request; an agentic pipeline's cost varies per query based on the model's own, potentially variable, retrieval decisions, making capacity planning and cost forecasting (Chapter 8 Topic 1's token-budgeting concerns, now compounded across a variable number of retrieval rounds) genuinely harder.
- **Debugging an agentic RAG failure requires distinguishing a retrieval-quality problem (Chapter 7's pipeline) from a triggering-decision problem (did the model correctly decide to retrieve, and correctly judge the result's sufficiency)** — these are different failure categories requiring different fixes, exactly the same failure-category discipline this notebook series has repeatedly emphasized (Chapter 8 Topic 3's faithfulness/relevance/correctness framework is directly relevant here too).
- **Monitoring:** tracking the distribution of how many retrieval calls agentic RAG actually makes per query, over time, reveals whether the model's retry behavior is well-calibrated (a small number of queries needing a second retrieval attempt) or poorly calibrated (many queries retrieving repeatedly without genuine benefit, or too few queries retrying when they should) — a direct, measurable signal distinct from simply monitoring whether retrieval was triggered at all (Chapter 9 Topic 7's discipline, now extended to retrieval *frequency* per query, not just occurrence).


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Static RAG vs agentic RAG isn't a strictly-better-or-worse choice — it's a genuine trade-off between predictability/cost-efficiency and flexibility/potential-quality**, and the right choice depends on the actual query distribution: if most real queries are well-served by a single, well-tuned retrieval pass (which Chapter 7's reranking and MMR work specifically aims to make as good as possible), agentic RAG's added flexibility may add cost without a corresponding, measurable quality benefit. If a meaningful fraction of queries genuinely benefit from iterative refinement (an initial retrieval that's clearly insufficient, warranting a refined follow-up query), agentic RAG's flexibility becomes worth its added cost.
- **Whether to allow unlimited retrieval retries or cap them explicitly:** an uncapped agentic loop risks the same runaway-cost and runaway-latency problem `max_steps` (Chapter 10 Topic 2) already exists to prevent for any multi-tool-call agent — agentic RAG specifically needs this same bound applied to retrieval retries in particular, not just tool calls generally, since a model that's convinced a result is insufficient could otherwise retry indefinitely.
- **Whether the decision to retrieve again should be left entirely to the model's own judgment, or backstopped by an explicit, measurable confidence signal** (mirroring Chapter 9 Topic 1's confidence-gated triggering strategy) — a hybrid approach, where the model's own judgment is the primary driver but a measured retrieval-quality signal (like a low top-1 relevance score) provides an additional, harder check on whether to retry, may be more reliable than trusting the model's self-assessment alone.


### 6. Alternatives and When to Use Each

- **Static RAG (Chapters 7-9's default approach):** the right choice when a single, well-tuned retrieval pass genuinely serves most real queries well — predictable cost and latency, and this project's own reranking and MMR investments (Chapter 7) specifically aim to make this the common, sufficient case.
- **Agentic RAG:** worth adopting specifically for query types where a measured fraction genuinely benefit from iterative retrieval refinement — not adopted wholesale just because it's more sophisticated-sounding, following this notebook series' own repeated evidence-based adoption discipline.
- **A hybrid approach — static RAG as the default, with an explicit, narrow escalation to an agentic retry loop only when a measured retrieval-quality signal is low:** combines static RAG's predictability for the common case with agentic RAG's flexibility reserved for the specific, harder cases that actually need it — directly analogous to Chapter 9 Topic 1's layered triggering strategy, now applied to retrieval iteration rather than retrieval occurrence.


### 7. Common Mistakes and Production Failures

- Adopting agentic RAG universally without measuring whether the added cost and latency actually produces a corresponding quality improvement for real queries — treating "more agentic" as automatically "better" without evidence.
- Not capping the number of retrieval retries an agentic loop can make, risking the same runaway cost and latency `max_steps` already exists to prevent for tool-calling generally.
- Trusting the model's own judgment about retrieval sufficiency without any measured validation, reproducing the same unvalidated-triggering-behavior risk Chapter 9 Topic 3 specifically measured and warned against.
- Conflating a retrieval-pipeline-quality problem with a triggering-decision problem when debugging an agentic RAG failure, wasting effort fixing the wrong layer.
- Not monitoring the actual distribution of retrieval-retry counts per query, missing a leading indicator of miscalibrated model behavior before it shows up as a broader cost or quality problem.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the fundamental difference between static RAG and agentic RAG?
  A: Static RAG follows a fixed control flow — retrieve, then generate, always in that order, always exactly once — external to the model's own reasoning. Agentic RAG gives the model itself a role in deciding whether to retrieve, how many times, and whether a retrieved result is sufficient or warrants another attempt, using the same tool-calling mechanics that let a model choose to invoke any other tool.

- Q: Does agentic RAG use a different retrieval algorithm than static RAG?
  A: No — the underlying retrieval pipeline (hybrid BM25 and dense retrieval, reranking, MMR) is completely unchanged. What differs is purely the control structure deciding when and how many times that unchanged pipeline gets invoked, and whether the model itself makes that decision rather than a fixed, external rule.

**Intermediate**

- Q: What genuinely new capability does agentic RAG add beyond Chapter 9 Topic 3's model-decided retrieval triggering?
  A: Chapter 9 Topic 3 demonstrated a model deciding whether to call a retrieval tool once. Agentic RAG additionally requires the model to evaluate the *quality* of what came back and decide whether to retrieve again, potentially with a refined query — an iterative capability beyond a single trigger-or-don't decision, using the same multi-step agent-loop mechanics that already support repeated tool calls for any reason.

- Q: Why is agentic RAG's cost and latency profile harder to predict than static RAG's?
  A: Static RAG has a fixed, bounded cost per request — exactly one retrieval pass and one generation call. Agentic RAG's cost varies per query, since the model itself decides how many retrieval rounds a given query needs, meaning capacity planning and cost forecasting must account for a variable, not fixed, number of pipeline invocations per request.

**Advanced**

- Q: Design a hybrid approach that captures agentic RAG's benefit without giving up static RAG's predictability for the common case.
  A: Use static RAG (a single retrieve-then-generate pass) as the default for every query, but add an explicit, measured escalation trigger — for example, if the retrieval pipeline's own top-1 relevance score falls below a validated threshold — that specifically invokes an agentic retry loop only for that narrower, harder subset of queries. This mirrors exactly Chapter 9 Topic 1's layered triggering strategy (cheap default, escalate only when a measured signal warrants it), now applied to deciding when iterative retrieval refinement is actually needed, rather than applying agentic RAG's added cost and complexity to every single query uniformly.

- Q: A teammate argues agentic RAG should replace this project's current static RAG pipeline entirely, since it's strictly more capable. How do you respond?
  A: More capable doesn't mean unconditionally better — agentic RAG's added flexibility comes with real, direct cost and latency increases, and its benefit only materializes for queries that genuinely need iterative refinement. This project's existing reranking and MMR investments (Chapter 7) specifically aim to make a single retrieval pass sufficient for most real queries; replacing the entire pipeline with an agentic loop would add cost uniformly, including for the many queries that don't actually need the extra flexibility. The right approach is measuring what fraction of real queries would genuinely benefit before deciding how broadly to adopt agentic retrieval, exactly the same evidence-based discipline this notebook series has applied to every other architectural decision.

**Scenario-based**

- Q: After adopting agentic RAG, monitoring shows a small fraction of queries triggering 4-5 retrieval rounds with no apparent improvement in the final answer's quality across those rounds. Diagnose.
  A: This is a signal the model's self-assessment of retrieval sufficiency may be miscalibrated for this specific query pattern — it's retrying without genuine cause, and each additional round adds cost without corresponding benefit. Rather than assuming this behavior is inherently correct because the model chose it, this should be measured against a labeled test set (following Chapter 10 Topic 7's evidence-based methodology) specifically covering cases where retrieval is genuinely sufficient on the first attempt, to determine whether the model's "insufficient, retry" judgment is well-calibrated or needs correction — likely via clearer system-prompt guidance about what constitutes a sufficient result, or a hard cap on retrieval retries per query if the underlying miscalibration can't be easily corrected through prompting alone.

**Follow-up questions to expect:**

- "How would you measure whether agentic RAG's added cost is actually justified for your specific query distribution?"
- "What would you cap the maximum number of retrieval retries at, and how would you decide that number?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Agentic RAG is a specific application of the general "give the model agency over a previously-fixed pipeline stage" pattern**, the same underlying idea behind every model-decided behavior already built in this notebook series — tool-calling generally (Chapter 10), retrieval triggering specifically (Chapter 9 Topic 1), and now retrieval *iteration* specifically. Recognizing agentic RAG as an instance of this recurring pattern, rather than a wholly separate new idea, connects it directly to everything already understood about model-decided behavior's benefits and risks.
- **The static-vs-agentic trade-off (predictability vs flexibility) is the same shape of trade-off already seen in Chapter 9 Topic 5's RAG chain patterns (Stuff's single-call predictability vs Map-Reduce/Refine/Map-Rerank's flexibility at higher cost)** — recognizing this recurring shape across multiple, seemingly different topics in this notebook series is a mark of genuinely transferable understanding, not memorized, disconnected facts.
- **This topic is the conceptual foundation for the rest of Chapter 13:** Topic 2's mechanics of letting the model decide, Topic 3's specific "check catalog only if uncertain" pattern, and Topic 4's Smart Saver FD validation are all concrete elaborations of the static-vs-agentic distinction established here.

### 10. Quick Revision Summary (for last-minute interview prep)

> Static RAG follows a fixed, external control flow — retrieve, then generate, always exactly once — regardless of a specific query's actual needs. Agentic RAG gives the model itself a role in that decision, using the exact same tool-calling mechanics already built for any other tool (Chapter 10): deciding whether to retrieve at all, evaluating whether a retrieved result is sufficient, and potentially retrieving again with a refined query if not — an iterative capability beyond Chapter 9 Topic 3's single trigger-or-don't decision. The underlying retrieval pipeline itself (Chapter 7's hybrid BM25+dense+RRF+reranking) is completely unchanged; only the control structure deciding when and how many times to invoke it differs. This flexibility carries a real, direct cost and latency trade-off compared to static RAG's predictable, bounded cost per request, and the model's own judgment about retrieval sufficiency is itself a probabilistic behavior needing the same evidence-based validation discipline as every other model-decided behavior in this notebook series. The right approach — mirroring Chapter 9 Topic 1's layered triggering strategy — is often a hybrid: static RAG as the default, with agentic iteration reserved specifically for queries where a measured signal indicates the first retrieval attempt was genuinely insufficient, rather than adopting agentic RAG universally without evidence that its added cost is actually justified.


### Module 1: Setup — A Knowledge Base and Two Query Types (Easy vs Genuinely Needs Refinement)

Build a real BM25-backed knowledge base and two test queries: one static RAG handles fine in one pass, one that genuinely benefits from query refinement.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- real knowledge base, two genuinely different query types
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return text.lower().split()

KNOWLEDGE_BASE = [
    {"text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.", "source": "01_FD_FAQ.pdf"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.", "source": "02_FD_Product_Guide.pdf"},
    {"text": "Step 3 of FD closure: apply the 1 percent penalty deduction on the interest rate before crediting the remaining amount.", "source": "03_FD_SOPs.pdf"},
    {"text": "Nomination facility is available for all Fixed Deposit account holders at no charge.", "source": "02_FD_Product_Guide.pdf"},
]
tokenized_corpus = [tokenize(doc["text"]) for doc in KNOWLEDGE_BASE]
bm25 = BM25Okapi(tokenized_corpus)


def retrieve(query: str, top_k: int = 1) -> list:
    """The REAL, unchanged retrieval pipeline (simplified BM25 stand-in
    for Chapter 7's full hybrid pipeline) -- IDENTICAL underneath for
    both static and agentic RAG. Only the control structure around
    this function differs between the two approaches."""
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(range(len(KNOWLEDGE_BASE)), key=lambda i: scores[i], reverse=True)
    return [{"text": KNOWLEDGE_BASE[i]["text"], "source": KNOWLEDGE_BASE[i]["source"], "score": scores[i]}
            for i in ranked[:top_k]]


# Query A: well-formed, first retrieval pass is genuinely sufficient
QUERY_EASY = "penalty premature withdrawal FD"

# Query B: vague, first retrieval pass is genuinely weak -- needs refinement
QUERY_HARD_INITIAL = "account rules general information"
QUERY_HARD_REFINED = "closure penalty deduction step interest"

print("=" * 70)
print("MODULE 1: KNOWLEDGE BASE AND TWO QUERY TYPES")
print("=" * 70)

easy_result = retrieve(QUERY_EASY)
print(f"\nQuery A (easy): '{QUERY_EASY}'")
_w101 = easy_result[0]["score"]
_w102 = easy_result[0]["text"]
print(f"  Top result: score={_w101:.4f} | {_w102[:60]}...")

hard_initial_result = retrieve(QUERY_HARD_INITIAL)
print(f"\nQuery B (vague, initial): '{QUERY_HARD_INITIAL}'")
_w103 = hard_initial_result[0]["score"]
_w104 = hard_initial_result[0]["text"]
print(f"  Top result: score={_w103:.4f} | {_w104[:60]}...")

print("\nQuery A's initial retrieval is already strong -- static RAG should")
print("handle it fine in one pass. Query B's initial retrieval is weak")
print("(low score, vague match) -- this is where agentic refinement could")
print("genuinely help, IF the model correctly recognizes insufficiency.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: KNOWLEDGE BASE AND TWO QUERY TYPES

Query A (easy): 'penalty premature withdrawal FD'
  Top result: score=1.7470 | Premature withdrawal of FD incurs a 1 percent penalty on the...

Query B (vague, initial): 'account rules general information'
  Top result: score=0.9014 | Nomination facility is available for all Fixed Deposit accou...

Query A's initial retrieval is already strong -- static RAG should
handle it fine in one pass. Query B's initial retrieval is weak
(low score, vague match) -- this is where agentic refinement could
genuinely help, IF the model correctly recognizes insufficiency.

Module 1 complete. Run Module 2 next.


### Module 2: Static RAG — Fixed, Single-Pass Control Flow

Implement the real static RAG control flow: exactly one retrieval call, regardless of query difficulty, measuring real cost (call count) for both query types.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Static RAG -- fixed, single-pass, no model involvement
# in the retrieval decision at all
# ------------------------------------------------------------------

def static_rag(query: str) -> dict:
    """STATIC RAG: exactly one retrieval call, always, regardless of
    the query's actual difficulty or the result's actual quality.
    The control flow is FIXED and external to any model reasoning."""
    retrieval_calls = 0
    result = retrieve(query)
    retrieval_calls += 1
    return {"final_context": result, "retrieval_calls_used": retrieval_calls}


print("=" * 70)
print("MODULE 2: STATIC RAG -- FIXED SINGLE-PASS CONTROL FLOW")
print("=" * 70)

static_easy = static_rag(QUERY_EASY)
print(f"\nQuery A (easy) via STATIC RAG:")
_v1 = static_easy["retrieval_calls_used"]
print(f"  Retrieval calls used: {_v1}")
_v2 = static_easy["final_context"]
_w105 = _v2[0]["score"]
print(f"  Final context score: {_w105:.4f}")

static_hard = static_rag(QUERY_HARD_INITIAL)
print(f"\nQuery B (vague) via STATIC RAG:")
_v3 = static_hard["retrieval_calls_used"]
print(f"  Retrieval calls used: {_v3}")
_v4 = static_hard["final_context"]
_w106 = _v4[0]["score"]
print(f"  Final context score: {_w106:.4f}")

print("\nSTATIC RAG used EXACTLY 1 retrieval call for BOTH queries --")
print("predictable, bounded cost, but Query B's weak result (low score)")
print("gets passed to generation AS-IS, with no opportunity to improve it.")
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: STATIC RAG -- FIXED SINGLE-PASS CONTROL FLOW

Query A (easy) via STATIC RAG:
  Retrieval calls used: 1
  Final context score: 1.7470

Query B (vague) via STATIC RAG:
  Retrieval calls used: 1
  Final context score: 0.9014

STATIC RAG used EXACTLY 1 retrieval call for BOTH queries --
predictable, bounded cost, but Query B's weak result (low score)
gets passed to generation AS-IS, with no opportunity to improve it.

Module 2 complete. Run Module 3 next.


### Module 3: Agentic RAG — Model Evaluates and Decides Whether to Retry

Implement a real evaluate-and-retry control loop, simulating the model's sufficiency judgment honestly, and measure the actual cost and quality difference against static RAG for both query types.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Agentic RAG -- REAL evaluate-and-retry control loop
# ------------------------------------------------------------------

SUFFICIENCY_THRESHOLD = 1.2  # calibrated against this corpus's actual BM25 score scale

def simulate_model_sufficiency_judgment(retrieved_result: list) -> bool:
    """Simulates the model's OWN judgment about whether a retrieval
    result is sufficient. In a real system this would be the model's
    own reasoning inside a generation call; here we use the retrieval
    score itself as an honest proxy, since we cannot call a real model
    to make this judgment offline. A score below threshold means the
    model would (correctly, in this simulation) judge it insufficient."""
    return retrieved_result[0]["score"] >= SUFFICIENCY_THRESHOLD


def agentic_rag(initial_query: str, refined_query: str = None, max_retries: int = 2) -> dict:
    """AGENTIC RAG: the model evaluates each retrieval result and
    decides whether to retry with a refined query -- a REAL loop,
    bounded by max_retries (mirroring Chapter 10 Topic 2's max_steps
    principle to prevent runaway cost)."""
    retrieval_calls = 0
    current_query = initial_query
    result = None

    for attempt in range(max_retries):
        result = retrieve(current_query)
        retrieval_calls += 1

        is_sufficient = simulate_model_sufficiency_judgment(result)
        if is_sufficient:
            break  # model judges this result good enough -- stop retrying

        # model decides to retry -- WITH a refined query, if one is available
        if refined_query is not None and current_query != refined_query:
            current_query = refined_query
        else:
            break  # no refinement available, or already tried it -- stop

    return {"final_context": result, "retrieval_calls_used": retrieval_calls,
            "final_query_used": current_query}


print("=" * 70)
print("MODULE 3: AGENTIC RAG -- REAL EVALUATE-AND-RETRY, MEASURED")
print("=" * 70)

agentic_easy = agentic_rag(QUERY_EASY)
print(f"\nQuery A (easy) via AGENTIC RAG:")
_v5 = agentic_easy["retrieval_calls_used"]
print(f"  Retrieval calls used: {_v5}")
_v6 = agentic_easy["final_context"]
_w107 = _v6[0]["score"]
print(f"  Final context score: {_w107:.4f}")

agentic_hard = agentic_rag(QUERY_HARD_INITIAL, refined_query=QUERY_HARD_REFINED)
print(f"\nQuery B (vague) via AGENTIC RAG (with refined follow-up query available):")
_v7 = agentic_hard["retrieval_calls_used"]
print(f"  Retrieval calls used: {_v7}")
_v8 = agentic_hard["final_query_used"]
print(f"  Final query actually used: '{_v8}'")
_v9 = agentic_hard["final_context"]
_w108 = _v9[0]["score"]
print(f"  Final context score: {_w108:.4f}")

print("\n" + "=" * 70)
print("STATIC vs AGENTIC -- SIDE-BY-SIDE COMPARISON")
print("=" * 70)
print(f"{'Query':<12} | {'Static calls':>12} | {'Static score':>12} | {'Agentic calls':>13} | {'Agentic score':>13}")
print("-" * 70)

static_easy_calls = static_easy["retrieval_calls_used"]
static_easy_score = static_easy["final_context"][0]["score"]
agentic_easy_calls = agentic_easy["retrieval_calls_used"]
agentic_easy_score = agentic_easy["final_context"][0]["score"]
print(f"{'A (easy)':<12} | {static_easy_calls:>12} | {static_easy_score:>12.4f} | "
      f"{agentic_easy_calls:>13} | {agentic_easy_score:>13.4f}")

static_hard_calls = static_hard["retrieval_calls_used"]
static_hard_score = static_hard["final_context"][0]["score"]
agentic_hard_calls = agentic_hard["retrieval_calls_used"]
agentic_hard_score = agentic_hard["final_context"][0]["score"]
print(f"{'B (vague)':<12} | {static_hard_calls:>12} | {static_hard_score:>12.4f} | "
      f"{agentic_hard_calls:>13} | {agentic_hard_score:>13.4f}")

if agentic_easy_calls == static_easy_calls:
    print("\nFor Query A (easy): agentic RAG used the SAME number of calls as")
    print("static RAG -- no wasted cost, since the model correctly judged the")
    print("first result sufficient. This is agentic RAG behaving WELL.")

if agentic_hard_calls > static_hard_calls and agentic_hard_score > static_hard_score:
    print("\nFor Query B (vague): agentic RAG used ONE MORE call than static RAG,")
    print("but achieved a MEASURABLY HIGHER final context score -- this extra")
    print("cost bought a genuine quality improvement, exactly the trade-off")
    print("this topic's theory describes.")

print("\nModule 3 complete. All Chapter 13 Topic 1 modules done.")
print()
print("=" * 70)
print("CHAPTER 13 TOPIC 1 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  The underlying retrieval pipeline (retrieve()) is IDENTICAL for both
  static and agentic RAG -- only the CONTROL STRUCTURE around it differs.

  Static RAG: fixed, exactly 1 call, always -- predictable cost,
  demonstrated with real, bounded call counts for both easy and hard
  queries.

  Agentic RAG: evaluate-and-retry, bounded by max_retries -- variable
  cost, demonstrated concretely: SAME cost as static for the easy query
  (no waste), MORE cost but MEASURABLY BETTER result for the hard query.

  This is a genuine trade-off, not a strict improvement -- the right
  choice depends on the ACTUAL query distribution and whether measured
  quality gains justify the added, variable cost.
""")


MODULE 3: AGENTIC RAG -- REAL EVALUATE-AND-RETRY, MEASURED

Query A (easy) via AGENTIC RAG:
  Retrieval calls used: 1
  Final context score: 1.7470

Query B (vague) via AGENTIC RAG (with refined follow-up query available):
  Retrieval calls used: 2
  Final query actually used: 'closure penalty deduction step interest'
  Final context score: 1.5991

STATIC vs AGENTIC -- SIDE-BY-SIDE COMPARISON
Query        | Static calls | Static score | Agentic calls | Agentic score
----------------------------------------------------------------------
A (easy)     |            1 |       1.7470 |             1 |        1.7470
B (vague)    |            1 |       0.9014 |             2 |        1.5991

For Query A (easy): agentic RAG used the SAME number of calls as
static RAG -- no wasted cost, since the model correctly judged the
first result sufficient. This is agentic RAG behaving WELL.

For Query B (vague): agentic RAG used ONE MORE call than static RAG,
but achieved a MEASURABLY HIGHER final contex